#  ⭐ `dim_dose_frequencia`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root,  normalize_dose_freq

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet") 
bronze =  bronze[["FREQUENCIA_DOSE"]].drop_duplicates()
bronze.columns

Index(['FREQUENCIA_DOSE'], dtype='object')

In [4]:
bronze.FREQUENCIA_DOSE.nunique()

1638

In [5]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_DOSE_FREQUENCIA.csv", sep=";", index=False)

In [6]:
bronze.head(40)

,FREQUENCIA_DOSE
0,6 hora
1,None
10,cíclico
12,4 hora
14,1 cíclico
16,1 dia
27,1
43,
65,12 hora
71,1 por 1 dia


# 🥈 Silver

normalized data, medium quality


In [7]:
silver = normalize_dose_freq(bronze, col="FREQUENCIA_DOSE")
silver.head()

,FREQUENCIA_DOSE,FREQUENCIA_DOSE_DOSES,FREQUENCIA_DOSE_INTERVALO_VALOR,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
0,6 hora,1.0,6.0,hora,3
1,None,NaN,NaN,desconhecido,0
10,cíclico,1.0,NaN,ciclico,9
12,4 hora,1.0,4.0,hora,3
14,1 cíclico,1.0,NaN,ciclico,9


In [8]:
silver[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE', 'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].drop_duplicates().sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR')

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
1,desconhecido,0
4545,segundo,1
4432,minuto,2
0,hora,3
16,dia,4
1037,semana,5
566,mes,6
19443,ano,7
281969,decada,8
10,ciclico,9


In [9]:
silver.FREQUENCIA_DOSE_INTERVALO_VALOR.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_INTERVALO_VALOR
1.0     232
NaN     118
2.0     117
8.0     115
6.0     104
3.0     100
4.0      98
12.0     81
21.0     52
15.0     47
Name: count, dtype: int64

In [10]:
silver.FREQUENCIA_DOSE_DOSES.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_DOSES
1.0     552
2.0     127
3.0      94
4.0      87
6.0      63
5.0      52
8.0      39
12.0     38
7.0      32
10.0     29
Name: count, dtype: int64

In [11]:
silver.query("FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE == 'desconhecido'").FREQUENCIA_DOSE.value_counts(dropna=False).head(40)

FREQUENCIA_DOSE
None                      1
1                         1
                          1
3                         1
2                         1
 se necessário            1
4                         1
1 se necessário           1
4 se necessário           1
12                        1
1 por 1                   1
6                         1
7                         1
01 se necessário          1
01                        1
18                        1
8 se necessário           1
03                        1
5                         1
8                         1
1 por 1 se necessário     1
09                        1
30                        1
0                         1
25                        1
500                       1
2 se necessário           1
85                        1
3 se necessário           1
10                        1
4 por 2 se necessário     1
5 se necessário           1
6 se necessário           1
01 por 1 se necessário    1
15 se necessário          1
16  

In [12]:
hist = silver

In [13]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia.parquet", index=False)

In [14]:
hist.to_csv(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia_MANUAL.csv", sep=";",index=False)

# 🥇 Gold


In [15]:
hist.columns

Index(['FREQUENCIA_DOSE', 'FREQUENCIA_DOSE_DOSES',
       'FREQUENCIA_DOSE_INTERVALO_VALOR',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR'],
      dtype='object')

In [16]:
dim = hist[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR').drop_duplicates()
dim

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
587211,desconhecido,0
256651,segundo,1
565281,minuto,2
543054,hora,3
98065,dia,4
2017,semana,5
581437,mes,6
220221,ano,7
281969,decada,8
10,ciclico,9


In [17]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_dose_frequencia/dim_dose_frequencia.parquet", index=False)